# Generate tool-use SFT traces

Pipeline:
1. **Pass 1** — DeepSeek generates batches of 10 questions per category in JSON mode (questions + steps, no results, no final).
2. **Tool execution** — Python runs the real tool calls in parallel and fills in the actual Result strings. Filters out invalid traces.
3. **Pass 2** — DeepSeek writes a grounded Final answer for each trace given the real tool results.
4. **Assemble** — Build the canonical trace text and write `tool_traces_grounded.jsonl`.

**Before running:**
- Runtime → CPU is fine (no model load here, only API calls + tool execution)
- Set `DEEPSEEK_API_KEY` in Colab Secrets
- Ensure `MyDrive/sparkyllm/datasets_sft/tool_traces/` is writable (it'll be created if missing)

Estimated cost: ~$1–2 in DeepSeek API spend. Estimated time: ~15–30 minutes (web_search calls dominate).

## 1. Setup

In [ ]:
!pip install -q openai

In [ ]:
import os, json, time, random, logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from google.colab import drive, userdata
from tqdm import tqdm

logging.getLogger('openai').setLevel(logging.WARNING)
logging.getLogger('httpx').setLevel(logging.WARNING)

drive.mount('/content/drive')

SPARKYLLM = '/content/drive/MyDrive/sparkyllm'
OUT_DIR = os.path.join(SPARKYLLM, 'datasets_sft', 'tool_traces')
os.makedirs(OUT_DIR, exist_ok=True)

RAW_FILE       = os.path.join(OUT_DIR, 'tool_traces_raw.jsonl')
GROUNDED_FILE  = os.path.join(OUT_DIR, 'tool_traces_grounded.jsonl')
META_FILE      = os.path.join(OUT_DIR, 'generate_meta.json')

DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
assert DEEPSEEK_API_KEY, 'Set DEEPSEEK_API_KEY in Colab Secrets first'
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url='https://api.deepseek.com')
JUDGE_MODEL = 'deepseek-chat'
print('DeepSeek client ready')

## 2. Pull repo to import the real `tools.py`

We use the same `local_agent/tools.py` that the agent uses at inference time, so training data is grounded in the exact tool behaviour.

In [ ]:
import subprocess
REPO_DIR = '/content/sparkyllm'
if not os.path.exists(REPO_DIR):
    print('Cloning sparkyllm...')
    subprocess.run(['git', 'clone', 'https://github.com/ajencinas/sparkyllm.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

import sys
sys.path.insert(0, os.path.join(REPO_DIR, 'local_agent'))
import importlib
if 'tools' in sys.modules:
    importlib.reload(sys.modules['tools'])
import tools as agent_tools

print('Available tools:', list(agent_tools.TOOLS))

## 3. Configuration

In [ ]:
# Distribution: 3000 traces total. Over-generate by ~20% to absorb filter losses.
TARGET_DISTRIBUTION = {
    'single_calculator': 750,
    'single_web_search': 750,
    'single_time':       150,
    'single_none':       750,
    'multi_step':        600,
}
OVERGENERATE_FACTOR = 1.20  # generate 20% more raw traces to compensate for filter dropouts

BATCH_SIZE = 10  # questions per DeepSeek call
MAX_PARALLEL_TOOL_WORKERS = 8  # threads for parallel tool execution

GEN_TEMPERATURE = 0.85   # high diversity for generation
FINAL_TEMPERATURE = 0.4  # lower for grounded final-writing

# Show planned generation
print('Planned generation (with overgenerate factor):')
total_raw = 0
for cat, target in TARGET_DISTRIBUTION.items():
    raw = int(target * OVERGENERATE_FACTOR)
    n_calls = (raw + BATCH_SIZE - 1) // BATCH_SIZE
    total_raw += raw
    print(f'  {cat:25s} target={target:4d}  raw={raw:4d}  ({n_calls} calls)')
print(f'  TOTAL raw: {total_raw}')

## 4. Pass 1 — generate raw traces

Sends batched requests to DeepSeek in JSON mode. Resumable: skips categories whose raw count is already met if `RAW_FILE` exists.

In [ ]:
GEN_SYSTEM_PROMPT = "You are generating training data for a small language model that will learn to use tools via a ReAct-style format.\n\nThe available tools are:\n- calculator: arithmetic. Input is a Python arithmetic expression (numbers, + - * / // % **, parentheses, floats). Example input: \"0.15 * 240\"\n- time: returns current date/time. Input must be empty string.\n- web_search: looks up a topic on the web. Input is a short search query (3-8 words).\n- none: no tool needed; the model answers directly. Input must be empty string.\n\nYou will be asked to generate questions in a specific category. Return a JSON object with this exact structure:\n\n{\n  \"traces\": [\n    {\n      \"question\": \"...\",\n      \"category\": \"...\",\n      \"steps\": [\n        {\"thought\": \"...\", \"action\": \"...\", \"input\": \"...\"}\n      ]\n    }\n  ]\n}\n\nRules per category:\n- single_calculator: exactly 1 step with action=\"calculator\". Input must be a valid Python arithmetic expression.\n- single_web_search: exactly 1 step with action=\"web_search\". Input must be a 3-8 word search query.\n- single_time: exactly 1 step with action=\"time\". Input must be empty.\n- single_none: exactly 1 step with action=\"none\". Input must be empty. Question should be a greeting, opinion, or simple question that does not need a tool.\n- multi_step: exactly 2 steps. The first step uses any of the four actions. The second step uses any of the four actions; action=\"none\" means \"no second tool needed; the model just reasons about the first result before answering\".\n\nEach thought must be one short sentence explaining what the model is about to do. Generate diverse, realistic questions a real user might actually ask. Avoid duplicates within a batch."

In [ ]:
def generate_batch(category, count):
    user_msg = f'Generate {count} diverse questions in category "{category}". Return only the JSON object as specified.'
    if category == 'multi_step':
        user_msg += ' At least half of the multi-step traces should have action="none" as the second step (i.e. the model uses one tool, then reasons about the result).'
    for attempt in range(3):
        try:
            comp = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {'role': 'system', 'content': GEN_SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_msg},
                ],
                response_format={'type': 'json_object'},
                temperature=GEN_TEMPERATURE,
                max_tokens=3500,
            )
            raw = comp.choices[0].message.content
            data = json.loads(raw)
            traces = data.get('traces', [])
            # Validate basic shape
            ok_traces = []
            for t in traces:
                if not isinstance(t, dict): continue
                if 'question' not in t or 'steps' not in t: continue
                if not isinstance(t['steps'], list) or len(t['steps']) == 0: continue
                # Force category field
                t['category'] = category
                ok_traces.append(t)
            return ok_traces
        except Exception as e:
            if attempt == 2:
                print(f'  batch failed for {category}: {e}')
                return []
            time.sleep(2)
    return []

# Resume support: read existing raw counts
existing_counts = {cat: 0 for cat in TARGET_DISTRIBUTION}
if os.path.exists(RAW_FILE):
    with open(RAW_FILE) as f:
        for line in f:
            try:
                obj = json.loads(line)
                cat = obj.get('category')
                if cat in existing_counts:
                    existing_counts[cat] += 1
            except Exception:
                pass
    print(f'Resuming. Existing raw counts: {existing_counts}')

with open(RAW_FILE, 'a') as f_out:
    for category, target in TARGET_DISTRIBUTION.items():
        raw_target = int(target * OVERGENERATE_FACTOR)
        already = existing_counts[category]
        if already >= raw_target:
            print(f'  {category}: already have {already}, skipping')
            continue
        needed = raw_target - already
        n_calls = (needed + BATCH_SIZE - 1) // BATCH_SIZE
        print(f'  {category}: generating {needed} more in {n_calls} calls')
        for _ in tqdm(range(n_calls), desc=category, leave=False):
            traces = generate_batch(category, BATCH_SIZE)
            for t in traces:
                f_out.write(json.dumps(t, ensure_ascii=False) + '\n')
            f_out.flush()

# Final count
raw_count = sum(1 for _ in open(RAW_FILE))
print(f'\nPass 1 complete. {raw_count} raw traces in {RAW_FILE}')

## 5. Tool execution — fill in real Result lines

For each raw trace, runs each step's `(action, input)` against the real `tools.py` and records the actual output. Filters out:
- traces with unknown action names
- traces where any step's tool returned `ERROR: ...`
- traces with malformed step structure

`web_search` is parallelised across threads to keep the wall clock down.

In [ ]:
def run_tools_for_trace(trace):
    grounded_steps = []
    for step in trace.get('steps', []):
        action = (step.get('action') or '').lower().strip()
        input_val = step.get('input', '') or ''
        if not isinstance(input_val, str):
            input_val = str(input_val)
        if action not in agent_tools.TOOLS:
            return None  # bad action
        try:
            result = agent_tools.TOOLS[action](input_val)
        except Exception as e:
            return None
        if not isinstance(result, str):
            return None
        if result.startswith('ERROR:'):
            return None
        grounded_steps.append({
            'thought': (step.get('thought') or '').strip(),
            'action': action,
            'input': input_val,
            'result': result,
        })
    return {
        'question': trace['question'],
        'category': trace['category'],
        'steps': grounded_steps,
    }

raw_traces = []
with open(RAW_FILE) as f:
    for line in f:
        try:
            raw_traces.append(json.loads(line))
        except Exception:
            pass
print(f'Loaded {len(raw_traces)} raw traces')

grounded = []
failed = 0
with ThreadPoolExecutor(max_workers=MAX_PARALLEL_TOOL_WORKERS) as exe:
    futures = [exe.submit(run_tools_for_trace, t) for t in raw_traces]
    for fut in tqdm(as_completed(futures), total=len(futures), desc='running tools'):
        gt = fut.result()
        if gt is None:
            failed += 1
        else:
            grounded.append(gt)

print(f'\nGrounded: {len(grounded)} | filtered out: {failed}')

# Show category breakdown
from collections import Counter
cat_counts = Counter(t['category'] for t in grounded)
print('\nGrounded by category:')
for cat, target in TARGET_DISTRIBUTION.items():
    have = cat_counts.get(cat, 0)
    print(f'  {cat:25s} {have:4d} / {target} target')

## 6. Pass 2 — DeepSeek writes grounded finals

For each grounded trace, ask DeepSeek to write a Final answer that synthesises the **real** tool results into one or two natural-language sentences.

In [ ]:
FINAL_SYSTEM_PROMPT = "You are helping generate training data for a small language model. You will be shown a list of completed tool-use traces (Question + Thought/Action/Input/Result blocks). For each trace, write a Final answer that synthesizes the tool results into a natural-language response to the user's question.\n\nRules:\n- Each Final must be one or two sentences.\n- Use only information present in the Result lines or general world knowledge that supports the result.\n- Do not contradict the Result.\n- Do not invent extra facts.\n\nReturn a JSON object with this exact structure:\n{\"finals\": [\"final for trace 1\", \"final for trace 2\", ...]}\n\nThe order of finals must exactly match the order of traces in the input."

In [ ]:
def trace_to_text_for_final(trace, idx):
    block = [f'### Trace {idx}', f'Question: {trace["question"]}']
    for step in trace['steps']:
        block.append(f'Thought: {step["thought"]}')
        block.append(f'Action: {step["action"]}')
        block.append(f'Input: {step["input"]}')
        block.append(f'Result: {step["result"]}')
    return '\n'.join(block)

def write_finals_batch(batch):
    text = '\n\n'.join(trace_to_text_for_final(t, i+1) for i, t in enumerate(batch))
    user_msg = (
        f'Write a Final answer for each of the following {len(batch)} traces. '
        f'Return JSON: {{"finals": ["...", "...", ...]}}.\n\n'
        + text
    )
    for attempt in range(3):
        try:
            comp = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[
                    {'role': 'system', 'content': FINAL_SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_msg},
                ],
                response_format={'type': 'json_object'},
                temperature=FINAL_TEMPERATURE,
                max_tokens=2500,
            )
            raw = comp.choices[0].message.content
            data = json.loads(raw)
            finals = data.get('finals', [])
            if isinstance(finals, list) and len(finals) == len(batch):
                return [str(s).strip() for s in finals]
        except Exception as e:
            if attempt == 2:
                return None
            time.sleep(2)
    return None

# Resume: skip traces already in GROUNDED_FILE
done_questions = set()
if os.path.exists(GROUNDED_FILE):
    with open(GROUNDED_FILE) as f:
        for line in f:
            try:
                obj = json.loads(line)
                done_questions.add(obj['question'])
            except Exception:
                pass
    print(f'Resuming. {len(done_questions)} grounded traces already have finals.')

to_finalize = [t for t in grounded if t['question'] not in done_questions]
print(f'Writing finals for {len(to_finalize)} traces...')

with open(GROUNDED_FILE, 'a') as f_out:
    for i in tqdm(range(0, len(to_finalize), BATCH_SIZE), desc='writing finals'):
        batch = to_finalize[i:i+BATCH_SIZE]
        finals = write_finals_batch(batch)
        if finals is None:
            print(f'  batch starting at {i} failed')
            continue
        for trace, final in zip(batch, finals):
            obj = {
                **trace,
                'final': final,
            }
            f_out.write(json.dumps(obj, ensure_ascii=False) + '\n')
        f_out.flush()

# Final count
grounded_count = sum(1 for _ in open(GROUNDED_FILE))
print(f'\nPass 2 complete. {grounded_count} grounded traces in {GROUNDED_FILE}')

## 7. Save metadata

In [ ]:
from collections import Counter
cat_final = Counter()
with open(GROUNDED_FILE) as f:
    for line in f:
        try:
            cat_final[json.loads(line)['category']] += 1
        except Exception:
            pass

meta = {
    'source': 'DeepSeek (deepseek-chat) two-pass with Python tool grounding',
    'judge_model': JUDGE_MODEL,
    'target_distribution': TARGET_DISTRIBUTION,
    'overgenerate_factor': OVERGENERATE_FACTOR,
    'batch_size': BATCH_SIZE,
    'gen_temperature': GEN_TEMPERATURE,
    'final_temperature': FINAL_TEMPERATURE,
    'final_counts': dict(cat_final),
    'total_grounded': sum(cat_final.values()),
}
with open(META_FILE, 'w') as f:
    json.dump(meta, f, indent=2)

print(json.dumps(meta, indent=2))
print()
print('Next: run tokenize_tool_sft.ipynb')